# Trend Analytics Agent — interactive demo

A walkthrough of the **Investigator + Writer + Critic** three-agent team in this repo.

By the end of this notebook you will have:

1. Verified your Azure OpenAI credentials work from this kernel.
2. Built (or re-used) a small SQLite warehouse from the **UCI Online Retail II** dataset.
3. Run sample SQL against the warehouse so you know what's in it.
4. Called each of the 7 bounded analysis tools directly, by hand.
5. Run the **Investigator** agent, watched it pick tools step by step.
6. Run the **Writer** agent on the Investigator's findings.
7. Run the **Critic** agent on the Investigator's findings + the Writer's draft.
8. Run the full **team loop** end-to-end, where the Critic decides whether to ship, ask the Writer to revise, or send the Investigator back for more evidence.
9. Edited the brief and re-run the team with your own question.

Open every cell, read the comment above it, then run it. Most cells take < 10 seconds; the agent cells take ~30–60 seconds each, and the team-loop cell can take 1–3 minutes depending on how many rounds the Critic asks for.

---

## The team pattern, in one paragraph

The Investigator queries the warehouse with seven bounded tools and produces structured *findings* (no prose). The Writer turns those findings into a Markdown briefing. The Critic then reads both the findings and the draft, and chooses one of three verdicts:

- **approve** — ship it.
- **revise_writer** — facts are sound but the briefing has issues (missing citations, overreach, etc.); the Writer revises without new data.
- **revise_investigator** — findings are unsafe to publish (a number with no source in the tool log, a flagged movement that was never decomposed); the Investigator goes back for another pass.

The loop is capped at a small number of rounds so it can't run away.

---

## Prerequisites

You should already have:

- This repo cloned to your working directory.
- A Python 3.10+ environment with `pip install -e .` already run (see `AZURE_ML_SETUP.md`).
- `az login --use-device-code` completed in the same terminal that launched Jupyter, **or** an Azure ML compute instance with managed identity.
- A `.env` file with `OPENAI_ENDPOINT` and `CHAT_DEPLOYMENT` filled in (copy from `.env.example`).

If anything below fails, the cell explains what to do.


In [ ]:
# Make `src/` importable and load .env. Run me first.
import sys
from pathlib import Path

ROOT = Path.cwd()
# Allow running the notebook from project root or from a subdir.
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

print(f"project root: {ROOT}")
print(f"src on path:  {SRC}")


### Install the project into this kernel

If you're using the default Azure ML `jupyter_env` kernel (or any env where you haven't run `pip install -e .` from a terminal), this cell makes the kernel self-sufficient: it installs this project in editable mode, which pulls in `pandas`, `openpyxl`, `azure-identity`, `openai`, `httpx`, and `python-dotenv`.

Idempotent — re-running is a quick no-op once everything is in place.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
print("dependencies ready in this kernel")

## 1. Environment and credentials

The agents read configuration from environment variables. The `Config` dataclass shows what was actually picked up.

If `OPENAI_ENDPOINT` is empty: copy `.env.example` to `.env`, fill in the endpoint and chat deployment name from your Azure OpenAI resource, then restart this kernel.


In [ ]:
from config import get_config

cfg = get_config()
print(f"OPENAI_ENDPOINT  = {cfg.openai_endpoint}")
print(f"CHAT_DEPLOYMENT  = {cfg.chat_deployment}")
print(f"WAREHOUSE_DB     = {cfg.warehouse_db}")
print(f"RUNS_DIR         = {cfg.runs_dir}")
print(f"REFERENCE_DATE   = {cfg.reference_date or '(auto from warehouse)'}")


### Verify the Azure credential

`DefaultAzureCredential` walks a chain (environment vars → managed identity → `az login` cache → VS Code → ...). On an Azure ML compute instance the managed identity link usually wins. On a laptop you'll need `az login --use-device-code` first.

The cell below tries to acquire a token for the Cognitive Services scope. If it succeeds, the agent will succeed too.

In [ ]:
from azure.identity import DefaultAzureCredential

try:
    cred = DefaultAzureCredential()
    token = cred.get_token("https://cognitiveservices.azure.com/.default")
    print(f"OK — got a token, expires at epoch {token.expires_on}")
except Exception as e:
    print("FAILED to acquire a token. Most common fix: run `az login --use-device-code` in the terminal that launched this kernel, then restart the kernel.")
    print(f"\nError: {e}")


## 2. Build the warehouse

The `scripts/ingest.py` script downloads the UCI Online Retail II dataset (~45 MB, CC BY 4.0) and builds `data/warehouse.db` with one table (`sales`) and one view (`sales_v`). It's idempotent: re-running uses the cached `.xlsx`.

If `data/warehouse.db` already exists you can skip this cell. To force a rebuild, delete the file first.

In [ ]:
import subprocess, os, pathlib
from importlib import reload
import config

# Azure ML compute instances mount ~/cloudfiles/ as Azure Files (network FS).
# SQLite file-locking is unreliable on network filesystems, so if the warehouse
# path lands on a known network mount we transparently redirect it to local
# instance disk (/tmp). The cached .xlsx stays on cloudfiles so no re-download.
db_path_str = str(ROOT / cfg.warehouse_db)
if any(marker in db_path_str for marker in ("/cloudfiles/", "/mnt/batch/")):
    safe_path = "/tmp/azure_agent_simple/warehouse.db"
    pathlib.Path("/tmp/azure_agent_simple").mkdir(parents=True, exist_ok=True)
    os.environ["WAREHOUSE_DB"] = safe_path
    reload(config)
    cfg = config.get_config()
    print(f"network mount detected — redirected WAREHOUSE_DB to {safe_path}")

db_path = ROOT / cfg.warehouse_db if not cfg.warehouse_db.is_absolute() else cfg.warehouse_db

if db_path.exists():
    size_mb = db_path.stat().st_size / 1_000_000
    print(f"warehouse already present at {db_path} ({size_mb:.1f} MB) — skipping ingest")
else:
    print(f"building warehouse at {db_path} — this takes ~2 minutes the first time…")
    env = {**os.environ, "WAREHOUSE_DB": str(db_path)}
    result = subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "ingest.py")],
        capture_output=True, text=True, cwd=ROOT, env=env,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("ingest failed — see output above")

## 3. Sample queries to verify the data

These are plain SQL against `sales_v` so you can see the shape of the data the agent will be reasoning about. The agent itself never writes raw SQL — it only fills parameters in the bounded tools — but you, the human, can.

In [ ]:
from db import warehouse

with warehouse(db_path) as wh:
    n_rows, _ = wh.run_query("SELECT COUNT(*) AS n FROM sales_v")
    date_range, _ = wh.run_query(
        "SELECT MIN(invoice_date) AS first_day, MAX(invoice_date) AS last_day FROM sales_v"
    )
    n_countries, _ = wh.run_query("SELECT COUNT(DISTINCT country) AS n FROM sales_v")
    n_products, _ = wh.run_query("SELECT COUNT(DISTINCT stock_code) AS n FROM sales_v")

print(f"rows:        {n_rows[0]['n']:>10,}")
print(f"date range:  {date_range[0]['first_day']} → {date_range[0]['last_day']}")
print(f"countries:   {n_countries[0]['n']:>10,}")
print(f"products:    {n_products[0]['n']:>10,}")


In [ ]:
# Top 10 countries by revenue (lifetime). Useful sanity check —
# you'll see the UK dominate, which colours how the agent will decompose later.
with warehouse(db_path) as wh:
    rows, _ = wh.run_query("""
        SELECT country, ROUND(SUM(quantity * price), 2) AS revenue
        FROM sales_v
        GROUP BY country
        ORDER BY revenue DESC
        LIMIT 10
    """)
for r in rows:
    print(f"  {r['country']:<25} {r['revenue']:>15,.2f}")


In [ ]:
# Monthly revenue across the whole history. Eyeballing this will tell
# you what the agent is comparing when it asks for current_month vs prior_month.
with warehouse(db_path) as wh:
    rows, _ = wh.run_query("""
        SELECT invoice_month, ROUND(SUM(quantity * price), 2) AS revenue
        FROM sales_v
        GROUP BY invoice_month
        ORDER BY invoice_month
    """)
for r in rows:
    print(f"  {r['invoice_month']}   {r['revenue']:>15,.2f}")


## 4. The seven tools the Investigator can call

The Investigator does not see the database directly. It sees seven function-call schemas. Each tool wraps one parameterised SQL query with a 5-second timeout and a 10k-row cap.

Listing the tools by name and one-line description.

In [ ]:
from tools import TOOL_SCHEMAS

for t in TOOL_SCHEMAS:
    fn = t["function"]
    desc = fn["description"].split(". ")[0]   # first sentence only
    print(f"- {fn['name']:<26} {desc}")


### Call a tool by hand — `metric_overview`

This is exactly what the Investigator does on its first turn: get current vs prior values for the headline metrics. We're calling the same Python method, with the same arguments shape, so you can see the structure the agent works with.

In [ ]:
import json
from db import warehouse
from tools import AnalysisTools

with warehouse(db_path) as wh:
    ref_date = cfg.reference_date or wh.reference_date()
    print(f"reference_date = {ref_date} (the latest date in the warehouse)")

    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)

    result = tools.metric_overview(
        metrics=["revenue", "units", "orders"],
        current_period={"name": "current_month"},
        comparison_period={"name": "prior_month"},
    )

print(json.dumps(result, indent=2, default=str))


### `dimension_decomposition`

Break revenue by country for the current month. The agent uses this to find where movement is concentrated.

In [ ]:
with warehouse(db_path) as wh:
    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)
    result = tools.dimension_decomposition(
        metric="revenue",
        dimension="country",
        period={"name": "current_month"},
        top_n=5,
    )
print(json.dumps(result, indent=2, default=str))


### `top_contributors`

Rank countries by their **signed contribution** to the change in revenue, current month vs prior. Positive values added to revenue; negative values subtracted from it.

In [ ]:
with warehouse(db_path) as wh:
    tools = AnalysisTools(warehouse=wh, reference_date=ref_date)
    result = tools.top_contributors(
        metric="revenue",
        dimension="country",
        period_a={"name": "current_month"},
        period_b={"name": "prior_month"},
        top_n=5,
    )
print(json.dumps(result, indent=2, default=str))


## 5. Run the Investigator

Now we let the agent drive. We pass `verbose=True` so each tool call prints as it happens — that's the "watch the agent think" view.

This cell makes a real Azure OpenAI request and will take ~30–60 seconds depending on how many tool calls the model decides to make (capped at 15).

In [ ]:
from agents.investigator import Investigator

# We open the warehouse for the duration of the agent run, then close it.
with warehouse(db_path) as wh:
    ref_date = cfg.reference_date or wh.reference_date()
    investigator = Investigator(
        openai_endpoint=cfg.openai_endpoint,
        warehouse=wh,
        reference_date=ref_date,
        chat_deployment=cfg.chat_deployment,
        verbose=True,
    )
    print(f"reference_date = {ref_date}, deployment = {cfg.chat_deployment}\n")
    print("--- agent tool calls ---")
    result = investigator.investigate()
    print(f"\n--- done in {result.iterations} iteration(s), {len(result.tool_calls)} tool call(s) ---")


### Findings — what the Investigator concluded

The Investigator's output is structured JSON, not prose. Each finding cites the `tool_call_id`s that support it.

In [ ]:
print(json.dumps({"summary": result.summary, "findings": result.findings}, indent=2))


## 6. Run the Writer

The Writer has **no tools** and **no database access**. Its only inputs are:

- the Investigator's findings JSON,
- the full tool-call log.

That's the structural reason the Writer cannot fabricate numbers. There's no path for it to invent a value — it can only quote what's already in the log.

In [ ]:
from agents.writer import write_briefing
from IPython.display import Markdown, display

briefing_md = write_briefing(
    openai_endpoint=cfg.openai_endpoint,
    chat_deployment=cfg.chat_deployment,
    reference_date=ref_date,
    findings=result.findings,
    summary=result.summary,
    tool_calls=result.tool_calls,
)

display(Markdown(briefing_md))


## 7. Run the Critic on the draft

The Critic closes the loop. It sees:

- the Investigator's structured findings + summary,
- the full tool-call log (every call and its result),
- the Markdown briefing the Writer just produced.

It does **not** have its own tools — like the Writer, it cannot fabricate or correct a number. It can only judge what the other two agents already produced and decide what happens next:

- **approve** — the briefing is solid; ship it.
- **revise_writer** — the facts are sound but the briefing has wording / citation issues. The Writer revises with no new data.
- **revise_investigator** — the findings themselves are unsafe (a fabricated number, a missed decomposition, an over-classified trend). The Investigator goes back for another pass.

Before calling the LLM, the Critic runs a deterministic check: it pulls every number out of the briefing and looks for a verbatim match in the tool-call log. The mismatched list is fed to the LLM as evidence — the LLM still makes the call (a 12.3 in the briefing vs 12.345 in the source is legitimate rounding, not a fabrication).

In [ ]:
from agents.critic import Critic

critic = Critic(
    openai_endpoint=cfg.openai_endpoint,
    chat_deployment=cfg.chat_deployment,
    verbose=True,
)

decision = critic.review(
    reference_date=ref_date,
    findings=result.findings,
    summary=result.summary,
    tool_calls=result.tool_calls,
    briefing=briefing_md,
)

print(f"verdict:  {decision.verdict}")
print(f"feedback: {decision.feedback}\n")

if decision.numbers_unsupported_raw:
    print(f"deterministic check flagged {len(decision.numbers_unsupported_raw)} number(s) as not-verbatim in the tool log:")
    print(f"  raw flagged:  {decision.numbers_unsupported_raw}")
    print(f"  judged ok:    {decision.numbers_judged_ok}    (likely rounding)")
    print(f"  judged bad:   {decision.numbers_judged_bad}   (real fabrications, if any)")
else:
    print("deterministic check: every number in the briefing appears verbatim in the tool log")

if decision.issues:
    print(f"\nissues raised ({len(decision.issues)}):")
    for i in decision.issues:
        print(f"  - [{i.get('kind','?')}] {i.get('where','?')}: {i.get('detail','')}")


## 8. Run the full team loop

The single-pass walkthrough above shows each agent in isolation. In production you don't run them by hand — `run_team()` chains them and lets the Critic drive revisions until either it approves or a hard cap on rounds is hit.

Each round looks like:

1. **Investigator** runs (initial pass, or a revision pass with the Critic's feedback baked into the brief).
2. **Writer** drafts (or revises) the briefing.
3. **Critic** approves, asks the Writer to revise, or asks the Investigator to dig deeper.

If the Critic asks for `revise_writer`, the next round skips the Investigator (no new tool calls) and just re-runs the Writer with feedback. If it asks for `revise_investigator`, the next round runs both. The tool-call log accumulates across rounds so the final briefing has full provenance.

`max_rounds=3` is a sensible default. Most runs settle in 1–2 rounds.

In [ ]:
from agents.team import run_team

with warehouse(db_path) as wh:
    investigator = Investigator(
        openai_endpoint=cfg.openai_endpoint,
        warehouse=wh,
        reference_date=ref_date,
        chat_deployment=cfg.chat_deployment,
        verbose=False,   # quieter — the team loop prints its own per-round summary
    )
    team_result = run_team(
        investigator=investigator,
        openai_endpoint=cfg.openai_endpoint,
        chat_deployment=cfg.chat_deployment,
        reference_date=ref_date,
        max_rounds=3,
        verbose=True,
    )

print(f"\napproved: {team_result.approved}")
print(f"rounds:   {len(team_result.rounds)}")
print(f"tool_calls (cumulative): {len(team_result.tool_calls)}")
print(f"findings: {len(team_result.findings)}")
print("\n--- per-round summary ---")
for r in team_result.rounds:
    inv = f"inv_iters={r.investigator_iterations}" if r.investigator_iterations is not None else "inv=skipped"
    print(f"  round {r.round}: verdict={r.verdict:<22} {inv}  new_tool_calls={r.new_tool_calls}")
    if r.feedback:
        print(f"           feedback: {r.feedback}")


### The final briefing — approved (or last draft, if max_rounds hit)

In [ ]:
display(Markdown(team_result.briefing))

## 9. Try your own brief — through the full team

Edit the brief below and re-run. The whole team runs against your question: Investigator → Writer → Critic, looping until the Critic approves or `max_rounds` is hit. Examples to try:

- `"Focus only on the United Kingdom — what changed?"`
- `"Compare this quarter to the same quarter last year."`
- `"Which products had the biggest declines this month?"`
- `"Is the daily revenue series trending up or down over the last 90 days?"`

If you deliberately steer the brief toward a tight filter (a single small country, a niche product), watch what the Critic does — that's where you're most likely to see a `revise_investigator` verdict trigger a second pass.

In [ ]:
my_brief = (
    "Focus on the United Kingdom this month vs last month. "
    "Identify the products that contributed most to the change."
)

with warehouse(db_path) as wh:
    investigator = Investigator(
        openai_endpoint=cfg.openai_endpoint,
        warehouse=wh,
        reference_date=ref_date,
        chat_deployment=cfg.chat_deployment,
        verbose=False,
    )
    custom = run_team(
        investigator=investigator,
        openai_endpoint=cfg.openai_endpoint,
        chat_deployment=cfg.chat_deployment,
        reference_date=ref_date,
        brief=my_brief,
        max_rounds=3,
        verbose=True,
    )

print(f"\napproved: {custom.approved}  rounds: {len(custom.rounds)}  tool_calls: {len(custom.tool_calls)}")
display(Markdown(custom.briefing))


## 10. Where to go from here

- **Modify a tool.** Edit `src/tools.py` and add a new bounded SQL tool — e.g. `month_over_month_seasonality`. Add its schema to `TOOL_SCHEMAS`. The agent will discover and use it on the next run.
- **Tighten the system prompts.** Edit `SYSTEM_PROMPT` in `src/agents/investigator.py` to change significance thresholds or stopping rules; edit it in `src/agents/critic.py` to change what the Critic enforces (e.g. require time-series evidence for any "trend" classification).
- **Tune the team loop.** Change `max_rounds` in `run_team()` or `scripts/run.py`. Lower it for tighter cost control; raise it if you find the Critic legitimately wants more passes.
- **Swap the dataset.** Replace `scripts/ingest.py` with one that loads your own CSV / SQL view as long as it produces a `sales_v` view with the same columns (`invoice_date`, `country`, `description`, `customer_id`, `quantity`, `price`).
- **Add a fourth role.** A *Distributor* agent that picks the audience (exec / analyst / ops) and asks the Writer for an audience-specific revision is a natural extension — it slots into the same loop pattern.
- **Schedule it.** Once you trust the output, wire `python scripts/run.py` into a weekly cron / Azure ML pipeline. Each run persists `briefing.md`, `findings.json`, `tool_calls.jsonl`, `critic.json`, and `run.json` under `data/runs/<run_id>/`.
